## Step 1: Set Up 

In [1]:
import sqlite3 
import pandas as pd 
import random

In [2]:
applicants = pd.read_csv(r"D:\NextGen Intern\applicants_cleaned dataset.csv")

In [3]:
applicants.head()

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Ayesha Malik,ayesha.malik01@gmail.com,3113356886,Web Development,Bahria University,4/20/2026,Under Review
1,Ali Khan,ali.khan02@outlook.com,3442868828,Data Science,University of Punjab,1/26/2026,Rejected
2,Hassan Raza,hassan.raza03@yahoo.com,3328728463,UI/UX Design,Air University,4/26/2026,Selected
3,Fatima Noor,fatima.noor04@example.com,3183197857,Mobile App Development,University of Lahore,1/26/2026,Selected
4,Usman Tariq,usman.tariq05@gmail.com,3189254563,Artificial Intelligence,Bahria University,5/2/2026,Under Review


In [ ]:
## create applicants_id

In [4]:
applicants.insert(
    0,
    "applicant_id",
    range(1,len(applicants)+1)
)

In [ ]:
## create interns table

In [5]:
selected = applicants[
    applicants["Status"] == "Selected"
].copy()

interns = selected.reset_index(drop=True)

interns.insert(
    0,
    "intern_id",
    range(1001, 1001 + len(interns))
)

interns["completion_status"] = [
    "Dropped Out" if i % 5 == 0 else "Completed"
    for i in range(len(interns))
]

interns["start_date"] = "2026-06-01"

interns = interns[
    ["intern_id", "applicant_id",
     "Domain Applied", "start_date",
     "completion_status"]
]

interns = interns.rename(
    columns={"Domain Applied": "domain"}
)



In [6]:
interns.head()

,intern_id,applicant_id,domain,start_date,completion_status
0,1001,3,UI/UX Design,2026-06-01,Dropped Out
1,1002,4,Mobile App Development,2026-06-01,Completed
2,1003,7,UI/UX Design,2026-06-01,Completed
3,1004,11,Data Science,2026-06-01,Completed
4,1005,12,UI/UX Design,2026-06-01,Completed


In [ ]:
## create hackathon table

In [7]:
completed = interns[
    interns["completion_status"] == "Completed"
].copy()

random.seed(42)

hackathon_scores = completed[
    ["intern_id", "domain"]
].copy()

hackathon_scores["score"] = [
    random.randint(55, 100)
    for _ in range(len(hackathon_scores))
]

hackathon_scores = hackathon_scores[
    ["intern_id", "score", "domain"]
]


In [8]:
hackathon_scores.head()

,intern_id,score,domain
1,1002,95,Mobile App Development
2,1003,62,UI/UX Design
3,1004,56,Data Science
4,1005,72,UI/UX Design
6,1007,70,Data Science


In [ ]:
## save all three table into SQLite

In [20]:
conn = sqlite3.connect("nextgen.db")

applicants.to_sql(
    "applicants",
    conn,
    if_exists="replace",
    index=False
)

interns.to_sql(
    "interns",
    conn,
    if_exists="replace",
    index=False
)

hackathon_scores.to_sql(
    "hackathon_scores",
    conn,
    if_exists="replace",
    index=False
)

print("All three tables created successfully!")

conn.close()

All three tables created successfully!


In [ ]:
## explore each table individually: 

In [23]:
conn = sqlite3.connect("nextgen.db")

In [24]:
pd.read_sql("SELECT * FROM applicants LIMIT 5;", conn)

,applicant_id,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,1,Ayesha Malik,ayesha.malik01@gmail.com,3113356886,Web Development,Bahria University,4/20/2026,Under Review
1,2,Ali Khan,ali.khan02@outlook.com,3442868828,Data Science,University of Punjab,1/26/2026,Rejected
2,3,Hassan Raza,hassan.raza03@yahoo.com,3328728463,UI/UX Design,Air University,4/26/2026,Selected
3,4,Fatima Noor,fatima.noor04@example.com,3183197857,Mobile App Development,University of Lahore,1/26/2026,Selected
4,5,Usman Tariq,usman.tariq05@gmail.com,3189254563,Artificial Intelligence,Bahria University,5/2/2026,Under Review


In [25]:
pd.read_sql("SELECT * FROM interns LIMIT 5;", conn) 

,intern_id,applicant_id,domain,start_date,completion_status
0,1001,3,UI/UX Design,2026-06-01,Dropped Out
1,1002,4,Mobile App Development,2026-06-01,Completed
2,1003,7,UI/UX Design,2026-06-01,Completed
3,1004,11,Data Science,2026-06-01,Completed
4,1005,12,UI/UX Design,2026-06-01,Completed


In [26]:
pd.read_sql("SELECT * FROM hackathon_scores LIMIT 5;", conn) 

,intern_id,score,domain
0,1002,95,Mobile App Development
1,1003,62,UI/UX Design
2,1004,56,Data Science
3,1005,72,UI/UX Design
4,1007,70,Data Science


## Step 3: Part A — Write and Explain 5 SQL Queries

In [ ]:
## Query 1 — How many interns completed each domain's program? 

In [28]:
## Counts completed interns grouped by domain

query1 = """
SELECT
    domain,
    COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""

result1 = pd.read_sql(query1, conn)

display(result1)

,domain,completed_count
0,UI/UX Design,6
1,Data Science,5
2,Web Development,3
3,Mobile App Development,3
4,Artificial Intelligence,3


In [ ]:
## Query 2 — What is the average hackathon score per domain? 

In [29]:
## Calculates the average hackathon score for each domain 

query2 = """
SELECT
    domain,
    ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""

result2 = pd.read_sql(query2, conn)

display(result2)

,domain,avg_score
0,Mobile App Development,84.33
1,Artificial Intelligence,81.67
2,UI/UX Design,75.17
3,Web Development,66.33
4,Data Science,62.20


In [ ]:
## Query 3 — Which interns scored above a threshold (top performers)? 

In [30]:
## Lists interns who scored 85 or above, as candidates for showcase/certificates 

query3 = """
SELECT
    i.intern_id,
    a."Applicant Name" AS intern_name,
    i.domain,
    h.score
FROM interns AS i
JOIN hackathon_scores AS h
    ON i.intern_id = h.intern_id
JOIN applicants AS a
    ON i.applicant_id = a.applicant_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""

result3 = pd.read_sql(query3, conn)

display(result3)

,intern_id,intern_name,domain,score
0,1012,Rehan Khalil,Mobile App Development,98
1,1002,Fatima Noor,Mobile App Development,95
2,1025,Shahbaz Ahmed,Artificial Intelligence,93
3,1015,Nimra Bashir,UI/UX Design,92
4,1013,Areeba Munir,Artificial Intelligence,89
5,1024,Danish Rauf,UI/UX Design,87


In [ ]:
## Query 4 — Conversion rate from "applied" to "completed" per domain

In [34]:
##  Compares total applicants vs completed interns per domain to find conversion rate 

query4 = """
SELECT
    a."Domain Applied" AS domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id)
        / COUNT(DISTINCT a.applicant_id),
        2
    ) AS conversion_rate_pct
FROM applicants AS a
LEFT JOIN interns AS i
    ON a.applicant_id = i.applicant_id
    AND i.completion_status = 'Completed'
GROUP BY a."Domain Applied"
ORDER BY conversion_rate_pct DESC;
"""

result4 = pd.read_sql(query4, conn)

display(result4)

,domain,total_applicants,total_completed,conversion_rate_pct
0,UI/UX Design,15,6,40.00
1,Data Science,16,5,31.25
2,Web Development,13,3,23.08
3,Mobile App Development,13,3,23.08
4,Artificial Intelligence,15,3,20.00


In [ ]:
## Query 5 — Your Choice (Required)

In [35]:
##  Which university produced the highest-scoring interns on average? 

query5 = """
SELECT
    a."University" AS university,
    COUNT(DISTINCT i.intern_id) AS total_interns,
    ROUND(AVG(h.score), 2) AS average_score
FROM applicants AS a
JOIN interns AS i
    ON a.applicant_id = i.applicant_id
JOIN hackathon_scores AS h
    ON i.intern_id = h.intern_id
GROUP BY a."University"
ORDER BY average_score DESC;
"""

result5 = pd.read_sql(query5, conn)

display(result5)

,university,total_interns,average_score
0,Air University,3,84.0
1,University of Lahore,5,79.6
2,Bahria University,5,70.8
3,University of Malakand,1,68.0
4,University of Punjab,4,65.5
5,Not Specified,2,62.5


## Step 4: Part B — Build the Dashboard 

### Option 1: Streamlit (Recommended if comfortable with Python) 